## 1. Project Introduction

# Loan Approval Prediction System

This project builds a Machine Learning model to predict whether a loan application should be approved or rejected.

The model is trained using the German Credit Dataset and multiple machine learning algorithms.

Models trained in this project:
- Logistic Regression (with Feature Scaling)
- Decision Tree
- Random Forest

The trained models and preprocessing components are saved and later used by a Streamlit application for real-time predictions.

Author: Adil Khan

## 2.Import Required Libraries

In [12]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

## 3.Load The Dataset

In [13]:
data = pd.read_csv("../data/loan_data.csv")

print("Dataset Loaded Successfully")
print("Shape:", data.shape)

data.head()

Dataset Loaded Successfully
Shape: (1000, 11)


,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,1,22,female,2,own,little,moderate,5951,48,radio/TV,bad
2,2,49,male,1,own,little,NaN,2096,12,education,good
3,3,45,male,2,free,little,little,7882,42,furniture/equipment,good
4,4,53,male,2,free,little,little,4870,24,car,bad


## 4.Data Cleaning

In [14]:
if 'Unnamed: 0' in data.columns:
    data.drop('Unnamed: 0', axis=1, inplace=True)

data['Loan_Status'] = data['Risk'].map({'good': 0, 'bad': 1})

data['Saving accounts'].fillna('none', inplace=True)
data['Checking account'].fillna('none', inplace=True)

data.head()

C:\Users\adilk\AppData\Local\Temp\ipykernel_20568\1844269372.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Saving accounts'].fillna('none', inplace=True)
C:\Users\adilk\AppData\Local\Temp\ipykernel_20568\1844269372.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk,Loan_Status
0,67,male,2,own,none,little,1169,6,radio/TV,good,0
1,22,female,2,own,little,moderate,5951,48,radio/TV,bad,1
2,49,male,1,own,little,none,2096,12,education,good,0
3,45,male,2,free,little,little,7882,42,furniture/equipment,good,0
4,53,male,2,free,little,little,4870,24,car,bad,1


## 5.Encode Categorical Features

In [15]:
categorical_columns = [
    'Sex',
    'Housing',
    'Saving accounts',
    'Checking account',
    'Purpose'
]

encoders = {}

for col in categorical_columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoders[col] = le

    print(f"{col} classes:", list(le.classes_))

Sex classes: ['female', 'male']
Housing classes: ['free', 'own', 'rent']
Saving accounts classes: ['little', 'moderate', 'none', 'quite rich', 'rich']
Checking account classes: ['little', 'moderate', 'none', 'rich']
Purpose classes: ['business', 'car', 'domestic appliances', 'education', 'furniture/equipment', 'radio/TV', 'repairs', 'vacation/others']


## 6.Define Features And Target Variables

In [16]:
X = data.drop(['Risk', 'Loan_Status'], axis=1)
y = data['Loan_Status']

X.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,67,1,2,1,2,0,1169,6,5
1,22,0,2,1,0,1,5951,48,5
2,49,1,1,1,0,2,2096,12,3
3,45,1,2,0,0,0,7882,42,4
4,53,1,2,0,0,0,4870,24,1


## 7.Save Feature Columns

In [17]:
feature_columns = list(X.columns)

print("Feature columns:", feature_columns)

joblib.dump(feature_columns, "../models/feature_columns.pkl")

Feature columns: ['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose']


['../models/feature_columns.pkl']

## 8.Train Test Split

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 9.Logistic Regression

In [19]:
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000))
])

lr_pipeline.fit(X_train, y_train)

y_pred_lr = lr_pipeline.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.73
              precision    recall  f1-score   support

           0       0.77      0.88      0.82       140
           1       0.57      0.38      0.46        60

    accuracy                           0.73       200
   macro avg       0.67      0.63      0.64       200
weighted avg       0.71      0.73      0.71       200



## 10.Decision Tree Model

In [20]:
dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

Decision Tree Accuracy: 0.66
              precision    recall  f1-score   support

           0       0.75      0.78      0.76       140
           1       0.43      0.38      0.40        60

    accuracy                           0.66       200
   macro avg       0.59      0.58      0.58       200
weighted avg       0.65      0.66      0.65       200



## 11.Random Forest Model

In [21]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.765
              precision    recall  f1-score   support

           0       0.80      0.89      0.84       140
           1       0.64      0.48      0.55        60

    accuracy                           0.77       200
   macro avg       0.72      0.68      0.70       200
weighted avg       0.75      0.77      0.75       200



## 12.Save The Trained Model

In [22]:
joblib.dump(lr_pipeline, "../models/loan_lr.pkl")
joblib.dump(dt_model, "../models/loan_dt.pkl")
joblib.dump(rf_model, "../models/loan_rf.pkl")

['../models/loan_rf.pkl']